In [0]:
import json
import requests
from datetime import datetime, timezone

CATALOG = "f1"
LANDING = f"/Volumes/f1/bronze/landing"

TESTS = {
    "jolpica": "https://api.jolpi.ca/ergast/f1/2025/1/results.json",
    "openf1": "https://api.openf1.org/v1/sessions?year=2025&session_name=Race",
}

In [0]:
results = {}
for name, url in TESTS.items():
    try: 
        r = requests.get(url)
        r.raise_for_status()
        results[name] = r.json()
        print(f"OK {name}: HTTP {r.status_code}, {len(r.content):,} bytes")
    except requests.exceptions.RequestException as e:
        print(f"FAIL {name}: {type(e).__name__}: {e}")

if "jolpica" in results:
    race = results["jolpica"]["MRData"]["RaceTable"]["Races"][0]
    winner = race["Results"][0]["Driver"]
    print(f"Jolpica: {race['raceName']} {race['season']}, winner {winner['givenName']} {winner['familyName']}")

if "openf1" in results:
    sessions = results["openf1"]
    print(f"OpenF1: {len(sessions)} race sessions in 2025, first: {sessions[0]['location']} (session_key {sessions[0]['session_key']})")

In [0]:
ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
for name, payload in results.items():
    path = f"{LANDING}/_connectivity_test/{name}_{ts}.json"
    dbutils.fs.mkdirs(path.rsplit("/", 1)[0])
    with open(path, "w") as f:
        json.dump(payload, f)
    print(f"Wrote {path}")
 
display(dbutils.fs.ls(f"{LANDING}/_connectivity_test/"))